# MAMMQA — Paper-Faithful Local Pipeline (All Gaps Fixed)
### T4 x2 Kaggle | Qwen2.5-VL-7B-Instruct | No API Keys

**Gaps Fixed vs Previous Version:**
- ✅ Gap 1+4: Qwen2.5-VL-7B (true VLM, processes images natively in all agents)
- ✅ Gap 2: temperature=0.3, top_p=0.7 (exact paper settings)
- ✅ Gap 3: ManymodalQA dataset support added
- ✅ Gap 5: Robustness testing (text shuffle + irrelevant context injection)
- ✅ Gap 6: Lightweight Tree-of-Thoughts comparison
- ✅ Gap 7: Question-agnostic aggregator ablation

**Kaggle Settings**: Accelerator = T4 x2, Internet = ON

In [1]:
# ============================================================
# Cell 1 — Install dependencies (torchvision version-matched)
# ============================================================
import sys, subprocess

def run(cmd):
    print(">", cmd[:90])
    subprocess.check_call(cmd, shell=True)

# Remove conflicting Pillow versions
run(f"{sys.executable} -m pip uninstall -y Pillow 2>/dev/null || true")

# Force install matching torch (2.5.1) and torchvision (0.20.1) for cu121
run(f"{sys.executable} -m pip install -q --no-cache-dir torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")

# Install other packages
run(
    f"{sys.executable} -m pip install -q --no-cache-dir "
    "\"pillow==10.4.0\" "
    "\"git+https://github.com/huggingface/transformers\" "
    "\"accelerate>=0.26.0\" "
    "\"bitsandbytes>=0.43.0\" "
    "\"qwen-vl-utils>=0.0.8\" "
    "\"sentencepiece\" "
    "\"pandas\" "
    "\"tqdm\" "
    "\"tabulate\" "
    "\"requests\" "
    "\"packaging\""
)
print("\n✅ Done. Restart kernel now — Runtime → Restart Session → Run All")

> /usr/bin/python3 -m pip uninstall -y Pillow 2>/dev/null || true
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
> /usr/bin/python3 -m pip install -q --no-cache-dir torch==2.5.1 torchvision==0.20.1 --index
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 306.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 289.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 339.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 336.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 248.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 296.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 327.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 336.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 316.4 MB/s eta 0:00:00
 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


> /usr/bin/python3 -m pip install -q --no-cache-dir "pillow==10.4.0" "git+https://github.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 298.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 389.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 279.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 277.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.



✅ Done. Restart kernel now — Runtime → Restart Session → Run All


# RESTART NEEDED

In [2]:
# ============================================================
# Cell 2 — Imports with fallback for qwen_vl_utils
# ============================================================
import sys, os, re, gc, json, time, random, shutil
from pathlib import Path
from typing import Dict, List, Optional, Union, Any
from collections import Counter

import requests
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

# Try qwen_vl_utils — if it fails, use a minimal fallback
QWEN_VL_UTILS_OK = False
try:
    from qwen_vl_utils import process_vision_info
    QWEN_VL_UTILS_OK = True
    print("✅ qwen_vl_utils loaded")
except Exception as e:
    print(f"⚠️  qwen_vl_utils failed ({type(e).__name__}): {e}")
    print("   Using PIL-based fallback for image processing...")

    def process_vision_info(messages):
        """Minimal fallback — extracts PIL images from file:// paths."""
        images = []
        for msg in messages:
            content = msg.get("content", [])
            if isinstance(content, list):
                for item in content:
                    if item.get("type") == "image":
                        img_ref = item.get("image", "")
                        if img_ref.startswith("file://"):
                            path = img_ref[7:]
                            if Path(path).exists():
                                images.append(Image.open(path).convert("RGB"))
        return images if images else None, None

    QWEN_VL_UTILS_OK = False
    print("   Fallback process_vision_info defined ✅")

import importlib.metadata as importlib_metadata

print("✅ Imports done")
print("PyTorch:", torch.__version__)
print("Transformers:", importlib_metadata.version("transformers"))
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    cap  = torch.cuda.get_device_capability(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | sm_{cap[0]}{cap[1]} | {vram:.1f}GB")

MODEL_NAME     = "Qwen/Qwen2.5-VL-7B-Instruct"
TEMPERATURE    = 0.3
TOP_P          = 0.7
MAX_NEW_TOKENS = 256
MAX_CHARS      = 3000

RUN_N_EXAMPLES                     = 20
SELECT_ONE_PER_TYPE                = True
REDUCE_MODALITIES_BY_QUESTION_TYPE = False
RUN_BASELINES                      = True
RUN_TOT                            = True
RUN_ROBUSTNESS                     = True
RUN_AGNOSTIC_ABLATION              = True
FETCH_WIKI_IMAGES                  = True

OUTPUT_DIR  = Path("/kaggle/working/mammqa_full_outputs")
IMAGE_CACHE = OUTPUT_DIR / "image_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Config ready | Output: {OUTPUT_DIR}")

✅ qwen_vl_utils loaded
✅ Imports done
PyTorch: 2.5.1+cu121
Transformers: 5.8.0.dev0
CUDA: True | GPUs: 2
  GPU 0: Tesla T4 | sm_75 | 15.6GB
  GPU 1: Tesla T4 | sm_75 | 15.6GB

✅ Config ready | Output: /kaggle/working/mammqa_full_outputs


In [3]:
# ============================================================
# Cell 3 — GPU verification
# ============================================================
assert torch.cuda.is_available(), "No GPU — enable T4 x2 in Kaggle settings"
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    assert cap[0] >= 7, f"GPU {i} is sm_{cap[0]}{cap[1]} — need T4 (sm_75) or better"

# Test ops that failed on P100
x = torch.tensor([1, -2, 3]).cuda()
assert (x < 0).any().item() == True
assert torch.isin(x, torch.tensor([1]).cuda()).any().item() == True
print("✅ All GPU ops verified — T4 fully working")

✅ All GPU ops verified — T4 fully working


In [4]:
# ============================================================
# Cell 4 — Dataset path discovery
# ============================================================

KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset"),
    Path("/kaggle/input/multimodalqa-dataset"),
    Path("/kaggle/input"),
]
EXPECTED_FILES = {
    "dev":    ["MMQA_dev.jsonl",    "dev.jsonl"],
    "texts":  ["MMQA_texts.jsonl",  "texts.jsonl"],
    "tables": ["MMQA_tables.jsonl", "tables.jsonl"],
    "images": ["MMQA_images.jsonl", "images.jsonl"],
}

def unwrap_kaggle_file(path: Path) -> Optional[Path]:
    if path.is_file(): return path
    if path.is_dir():
        files = [c for c in path.iterdir() if c.is_file()]
        if len(files) == 1: return files[0]
        for f in files:
            if f.suffix in ('.jsonl', '.json'): return f
    return None

def resolve_files() -> Dict[str, Optional[Path]]:
    root = next((c for c in KAGGLE_INPUT_CANDIDATES if c.exists()), None)
    if not root: print("❌ Dataset root not found"); return {}
    print(f"📂 Root: {root}")
    resolved = {}
    for key, names in EXPECTED_FILES.items():
        found = next((unwrap_kaggle_file(root/n) for n in names if unwrap_kaggle_file(root/n)), None)
        resolved[key] = found
        print(f"  {key:8s}: {'✅ '+str(found) if found else '❌ NOT FOUND'}")
    return resolved

FILE_PATHS  = resolve_files()
DEV_FILE    = FILE_PATHS["dev"]
TEXTS_FILE  = FILE_PATHS["texts"]
TABLES_FILE = FILE_PATHS["tables"]
IMAGES_FILE = FILE_PATHS["images"]
assert all([DEV_FILE, TEXTS_FILE, TABLES_FILE, IMAGES_FILE]), "Missing files!"
print("✅ All dataset files resolved")

📂 Root: /kaggle/input/datasets/adibraian/multimodalqa-dataset
  dev     : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_dev.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl
  texts   : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_texts.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl
  tables  : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_tables.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl
  images  : ✅ /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_images.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl
✅ All dataset files resolved


In [5]:
# ============================================================
# Cell 5 — Dataset loader
# ============================================================

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: data.append(json.loads(line))
                except: pass
    return data

def make_unique_cols(df):
    new, counts = [], {}
    for col in df.columns:
        counts[col] = counts.get(col, -1) + 1
        new.append(f"{col}_{counts[col]}" if counts[col] > 0 else col)
    df.columns = new
    return df

def parse_table(data):
    tb = data.get('table', {})
    headers = [h.get('column_name','') for h in tb.get('header',[])]
    rows = [[c.get('text','') for c in row] for row in tb.get('table_rows',[])]
    if rows:
        df = pd.DataFrame(rows, columns=headers) if headers and len(headers)==len(rows[0]) else pd.DataFrame(rows)
        return {"title": data.get('title',''), "markdown": make_unique_cols(df).to_markdown(index=False)}
    return {"title": data.get('title',''), "markdown": ""}

def parse_text(data):  return {"title": data.get('title',''),  "text":  data.get('text','')}
def parse_image(data): return {"title": data.get('title',''),  "path":  data.get('path','')}

print("Loading dataset (30-60 seconds)...")
dev_data    = load_jsonl(DEV_FILE)
texts_raw   = load_jsonl(TEXTS_FILE)
tables_raw  = load_jsonl(TABLES_FILE)
images_raw  = load_jsonl(IMAGES_FILE)

text_lookup  = {e['id']: parse_text(e)  for e in texts_raw  if e.get('id')}
table_lookup = {e['id']: parse_table(e) for e in tables_raw if e.get('id')}
image_lookup = {e['id']: parse_image(e) for e in images_raw if e.get('id')}

print(f"✅ {len(dev_data):,} questions | {len(text_lookup):,} texts | {len(table_lookup):,} tables | {len(image_lookup):,} images")

Loading dataset (30-60 seconds)...
✅ 2,441 questions | 218,285 texts | 10,042 tables | 57,058 images


In [6]:
# ============================================================
# Cell 6 — Build examples and fetch images
# ============================================================

def fetch_wiki_image(title):
    try:
        r = requests.get("https://en.wikipedia.org/w/api.php", params={
            "action":"query","titles":title,"prop":"pageimages",
            "pithumbsize":400,"format":"json"}, timeout=5)
        for p in r.json().get("query",{}).get("pages",{}).values():
            t = p.get("thumbnail",{}).get("source")
            if t: return t
    except: pass
    return None

def download_image(url, path):
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200: path.write_bytes(r.content); return path
    except: pass
    return None

def build_example(entry):
    meta     = entry.get('metadata', {})
    table_id = meta.get('table_id')
    text_ids = meta.get('text_doc_ids', [])
    img_ids  = meta.get('image_doc_ids', [])

    texts_data = [text_lookup[t] for t in text_ids if t in text_lookup]
    text_str   = "\n".join(f"title: {t['title']}\ntext: {t['text']}" for t in texts_data) or "No text"

    tbl = table_lookup.get(table_id)
    table_str = tbl['markdown'] if tbl else ""

    image_items = []
    for iid in img_ids:
        img = image_lookup.get(iid)
        if not img: continue
        cache = IMAGE_CACHE / f"{iid}.jpg"
        local = None
        if cache.exists(): local = cache
        elif FETCH_WIKI_IMAGES:
            url = fetch_wiki_image(img['title'])
            if url: local = download_image(url, cache)
        image_items.append({"title": img['title'], "local_path": str(local) if local else None})

    ans_list = entry.get('answers', [])
    first = ans_list[0] if ans_list else {}
    gold  = first.get('answer','') if isinstance(first, dict) else str(first)

    return {
        "id":          entry.get('qid', entry.get('id','')),
        "question":    entry.get('question',''),
        "gold_answer": gold,
        "type":        meta.get('type','unknown'),
        "modalities":  meta.get('modalities',[]),
        "text":        text_str,
        "table":       table_str,
        "image_items": image_items,
    }

# Select examples
target_types = ['TextQ','TableQ','ImageQ','Compose']
sel_indices, seen = [], set()
for i, e in enumerate(dev_data):
    qt = e.get('metadata',{}).get('type','')
    if SELECT_ONE_PER_TYPE:
        if qt in target_types and qt not in seen:
            seen.add(qt); sel_indices.append(i)
    else:
        sel_indices.append(i)
    if len(sel_indices) >= RUN_N_EXAMPLES: break

print(f"Building {len(sel_indices)} examples (fetching images)...")
examples = []
for i in tqdm(sel_indices):
    ex = build_example(dev_data[i])
    examples.append(ex)
    print(f"  [{ex['type']}] {ex['question'][:65]}")
    print(f"         Gold: {ex['gold_answer']}")

print(f"\n✅ {len(examples)} examples ready")

Building 3 examples (fetching images)...


  0%|          | 0/3 [00:00<?, ?it/s]

  [TableQ] For which film did Ben Piazza play the role of Mr. Simms?
         Gold: Mask
  [TextQ] What award did Barathea win at Churchill Downs in 1994?
         Gold: Breeders' Cup Mile
  [ImageQ] What animals race in the Kentucky Derby?
         Gold: horses

✅ 3 examples ready


In [7]:
# # ============================================================
# # Cell 7 — Load Qwen2.5-VL-7B-Instruct processor and model
# # ============================================================

# # ============================================================
# # Cell 7 — Load Qwen2.5-VL-7B-Instruct processor and model
# # ============================================================

# # Step 1: Ensure torchvision is version-COMPATIBLE (not just installed)
# # The original code caught ImportError only — but the real failure is a
# # RuntimeError during import when torchvision is compiled for a different
# # PyTorch version ("operator torchvision::nms does not exist").

# import subprocess, sys, importlib

# # PyTorch major.minor → (torchvision version, CUDA tag)
# TORCH_TV_MAP = {
#     "2.1": ("0.16.2", "cu121"),
#     "2.2": ("0.17.2", "cu121"),
#     "2.3": ("0.18.1", "cu121"),
#     "2.4": ("0.19.1", "cu121"),
#     "2.5": ("0.20.1", "cu121"),
#     "2.6": ("0.21.0", "cu124"),
# }
# torch_mm = ".".join(torch.__version__.split(".")[:2])
# tv_ver, cu_tag = TORCH_TV_MAP.get(torch_mm, ("0.20.1", "cu121"))

# def _torchvision_ok() -> bool:
#     """Returns True only if torchvision loads AND its operators bind correctly."""
#     try:
#         # Purge any cached broken import first
#         to_remove = [k for k in sys.modules if k.startswith("torchvision")]
#         for k in to_remove:
#             del sys.modules[k]
#         import torchvision
#         import torchvision.ops          # this is where nms registration fires
#         _ = torchvision.__version__     # confirm the object is sane
#         return True
#     except Exception as e:
#         print(f"   ↳ torchvision incompatible: {type(e).__name__}: {e}")
#         return False

# if _torchvision_ok():
#     import torchvision
#     print(f"✅ torchvision {torchvision.__version__} — fully compatible")
# else:
#     print(f"⚠️  torchvision version mismatch detected.")
#     print(f"   PyTorch {torch.__version__} requires torchvision=={tv_ver}")
#     print(f"   Force-reinstalling torchvision=={tv_ver} ({cu_tag})...")
#     subprocess.check_call([
#         sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
#         "--force-reinstall",
#         f"torchvision=={tv_ver}",
#         "--index-url", f"https://download.pytorch.org/whl/{cu_tag}",
#     ])
#     # Purge stale modules so the next import picks up the new build
#     to_remove = [k for k in sys.modules if k.startswith("torchvision")]
#     for k in to_remove:
#         del sys.modules[k]
#     import torchvision
#     print(f"✅ torchvision {torchvision.__version__} installed and loaded")

# # Step 2 and Step 3 stay exactly the same as before...

# # Step 2: Load processor
# gc.collect()
# torch.cuda.empty_cache()

# MIN_PIXELS = 256 * 28 * 28
# MAX_PIXELS = 512 * 28 * 28

# print(f"\nLoading processor: {MODEL_NAME}")
# try:
#     processor = AutoProcessor.from_pretrained(
#         MODEL_NAME,
#         min_pixels=MIN_PIXELS,
#         max_pixels=MAX_PIXELS,
#     )
#     print("✅ Processor loaded with video support")
# except Exception as e:
#     print(f"⚠️  Full processor failed ({e})")
#     print("   Loading image-only processor (no video — fine for our use case)...")
#     # Load components separately, skipping video processor
#     from transformers import Qwen2_5_VLImageProcessor, AutoTokenizer
#     from transformers import Qwen2_5_VLProcessor
#     img_proc   = Qwen2_5_VLImageProcessor.from_pretrained(
#         MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
#     )
#     tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
#     processor  = Qwen2_5_VLProcessor(
#         image_processor=img_proc,
#         tokenizer=tokenizer,
#     )
#     print("✅ Image-only processor loaded")

# # Step 3: Load model
# print(f"\nLoading model: {MODEL_NAME}")
# print("Expected time: 3-5 minutes (downloads ~4GB on first run)...")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
# )

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
#     low_cpu_mem_usage=True,
# )
# model.eval()

# gc.collect()
# torch.cuda.empty_cache()

# print(f"\n✅ Model loaded: {MODEL_NAME}")
# for i in range(torch.cuda.device_count()):
#     used  = torch.cuda.memory_allocated(i) / 1e9
#     total = torch.cuda.get_device_properties(i).total_memory / 1e9
#     print(f"   GPU {i}: {used:.1f}/{total:.1f} GB used")

In [8]:
# ============================================================
# Cell 7 — Load Qwen2.5-VL-7B-Instruct (Clean Load)
# ============================================================
import gc
import torch
import torchvision
from transformers import (
    AutoProcessor, 
    Qwen2_5_VLForConditionalGeneration, 
    BitsAndBytesConfig
)

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

# Verify environment
print(f"✅ torchvision {torchvision.__version__} — fully compatible")

# Step 1: Memory Cleanup
gc.collect()
torch.cuda.empty_cache()

# Step 2: Load Processor
print(f"\nLoading processor: {MODEL_NAME}")
try:
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
    print("✅ Processor loaded successfully")
except Exception as e:
    print(f"⚠️  Full processor failed: {e}. Attempting image-only fallback...")
    from transformers import Qwen2_5_VLImageProcessor, AutoTokenizer, Qwen2_5_VLProcessor
    img_proc = Qwen2_5_VLImageProcessor.from_pretrained(MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    processor = Qwen2_5_VLProcessor(image_processor=img_proc, tokenizer=tokenizer)
    print("✅ Image-only processor loaded")

# Step 3: Load Model with 4-bit Quantization
print(f"\nLoading model: {MODEL_NAME}")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

# Step 4: Final Memory Stats
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Model loaded: {MODEL_NAME}")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.1f}/{total:.1f} GB used")

✅ torchvision 0.20.1+cu121 — fully compatible

Loading processor: Qwen/Qwen2.5-VL-7B-Instruct


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

✅ Processor loaded successfully

Loading model: Qwen/Qwen2.5-VL-7B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 8 — MAMMQA prompts (Stage 3 updated for short answers)
# ============================================================

Agent_stage1_system_prompt = """
You are an expert agent specialized in analyzing single-modality inputs (such as text, table, or image) and answering questions by extracting insights and systematically breaking down complex questions into simpler subquestions.

### **Task**:
Step 1: Identify the modality type. Possible types: text(s), table(s), or image(s).
Step 2: Analyze the input and extract key insights (numbers, entities, temporal info).
Step 3: Break the main question into simpler subquestions.

## **Formatting**:
- <modality> type here </modality>
- <insights> insights here </insights>
- <answer> final answer here </answer>
- <subanswer> subquestion answer here </subanswer>
- Use ONLY provided data. No external knowledge.
"""

Agent_stage2_system_prompt = """
You are an expert cross-modal agent combining insights from specialist agents with multimodal inputs.

### **Task**:
Step 1: Analyze all inputs and insights. Extract key information.
Step 2: Break the question into subquestions using cross-modal evidence.
Note: If a modality is marked as UNAVAILABLE, do not treat it as contradictory evidence. Simply ignore it.

## **Formatting**:
- <insights> insights here </insights>
- <subanswer> subquestion answer here </subanswer>
- <answer> final answer here </answer>
- Use ONLY provided data. No external knowledge.
"""

# ── UPDATED Stage 3 prompt — forces short answer spans ──────────────────────
Agent_stage3_system_prompt = """
You are the final aggregator agent. You receive responses from cross-modal synthesis agents.

Rules for selecting the final answer:
(A) Consistency: If 2+ agents agree with clear reasoning, select that answer.
(B) Fallback: If some agents report unavailable modality, prefer the agent with real evidence.
(C) Conflict: If all differ, select the answer with the strongest supporting evidence.

CRITICAL OUTPUT RULE:
- Return ONLY the shortest answer span. Do NOT explain. Do NOT write a full sentence.
- Do NOT include words like "the film", "the answer is", "it is", etc.
- Output ONLY the answer entity, name, number, or word.

Examples of CORRECT output:
  <answer>Mask</answer>
  <answer>Breeders' Cup Mile</answer>
  <answer>horses</answer>
  <answer>1994</answer>

Examples of WRONG output (never do this):
  <answer>Ben Piazza played the role of Mr. Simms in the film Mask.</answer>
  <answer>The answer is Mask.</answer>
  <answer>Horses race in the Kentucky Derby.</answer>

Always end with exactly: <answer>short answer only</answer>
"""

print("✅ Prompts loaded — Stage 3 enforces short answer spans")

In [ ]:
# ============================================================
# Cell 9 — Core VLM inference + answer extraction
# ============================================================

def cap(text, n=MAX_CHARS):
    return (text[:n] + "...[capped]") if text and len(text) > n else (text or "")


def run_vl_model(
    system_prompt: str,
    user_text: str,
    image_paths: Optional[List[str]] = None,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    user_content = []
    if image_paths:
        for img_path in image_paths:
            if img_path and Path(img_path).exists():
                user_content.append({"type": "image", "image": f"file://{img_path}"})
    user_content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    has_images = any(c.get("type") == "image" for c in user_content)
    if has_images:
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text_input], images=image_inputs, videos=video_inputs,
            return_tensors="pt", truncation=True, max_length=5000,
        )
    else:
        inputs = processor(
            text=[text_input], return_tensors="pt",
            truncation=True, max_length=6000,
        )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    new_tokens = output_ids[0][input_len:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()


def extract_answer(text: str) -> str:
    """Extract content inside <answer>...</answer> tags only."""
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # Fallback: last non-empty line (avoid full paragraphs)
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[-1] if lines else text.strip()


def clean_short_answer(text: str) -> str:
    """
    Post-processing to shorten common verbose answer patterns.
    Handles cases where the model still returns a sentence despite the prompt.

    Examples:
      'Ben Piazza played the role of Mr. Simms in the film "Mask".' → 'Mask'
      'Barathea won the Breeders Cup Mile at Churchill Downs.'       → 'Breeders Cup Mile'
      'Horses race in the Kentucky Derby.'                           → 'horses'
    """
    t = text.strip().strip('"\'')

    # Pattern 1: "... in the film X" / "... called X" / "... named X" / "... is X"
    m = re.search(
        r'(?:in the (?:film|movie|show|series)|called|named|titled|is|was|are|were)\s+"?([^".]+)"?[.\s]*$',
        t, re.IGNORECASE
    )
    if m:
        candidate = m.group(1).strip().strip('"\'.,')
        # Only use this if it's meaningfully shorter (not just trimming whitespace)
        if len(candidate) < len(t) * 0.7:
            return candidate

    # Pattern 2: "X won the Y at Z" → extract Y
    m = re.search(r'\bwon (?:the\s+)?(.+?)(?:\s+at\s+|\s+in\s+|\.$)', t, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip().strip('"\'.,')
        if len(candidate) < len(t) * 0.7 and len(candidate) > 2:
            return candidate

    # Pattern 3: "X race/run/compete in the Y" → return Y
    m = re.search(r'(?:race|run|compete|play)\s+in\s+the\s+(.+?)[.\s]*$', t, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip().strip('"\'.,')
        if len(candidate) < len(t) * 0.7:
            return candidate

    # Pattern 4: ends with quoted term → extract it
    m = re.search(r'"([^"]+)"[.\s]*$', t)
    if m:
        return m.group(1).strip()

    # Pattern 5: if the text is a sentence (contains a verb phrase and is long),
    # take only the last quoted or capitalized noun phrase
    if len(t.split()) > 6 and re.search(r'\b(is|was|are|were|played|won|race)\b', t, re.IGNORECASE):
        # Try to get the last noun phrase — words before the period
        last_segment = re.split(r'[,;]', t)[-1].strip().rstrip('.')
        if len(last_segment.split()) <= 5 and len(last_segment) > 1:
            return last_segment

    return t


def extract_and_clean(raw: str) -> str:
    """Full pipeline: extract from tags, then clean if still verbose."""
    ans = extract_answer(raw)
    cleaned = clean_short_answer(ans)
    return cleaned


def get_image_paths(image_items: List[Dict]) -> List[str]:
    return [
        item['local_path'] for item in image_items
        if item.get('local_path') and Path(item['local_path']).exists()
    ]


print("✅ Inference functions ready")
print(f"   Temperature: {TEMPERATURE} | Top-p: {TOP_P} | Max new tokens: {MAX_NEW_TOKENS}")
print("   extract_and_clean() will shorten verbose answers automatically")

In [ ]:
# ============================================================
# Cell 10 — Stage 1, 2, 3 Agent functions
# Requirement 1: no question-type routing — data availability only
# Requirement 2: unavailable modalities marked, not skipped silently
# ============================================================

UNAVAILABLE_MSG = "[MODALITY UNAVAILABLE — no data provided. Do not treat as contradiction.]"

# ── STAGE 1 ──────────────────────────────────────────────────────────────────

def text_agent(q, texts):
    return run_vl_model(
        Agent_stage1_system_prompt,
        f"Here is the text data:\n{cap(texts)}\n\nHere is the question: {q}"
    )

def table_agent(q, tables):
    return run_vl_model(
        Agent_stage1_system_prompt,
        f"Here is the markdown table:\n{cap(tables)}\n\nHere is the question: {q}"
    )

def image_agent(q, image_items):
    img_paths  = get_image_paths(image_items)
    img_titles = ", ".join(i['title'] for i in image_items if i.get('title')) or "No images"
    return run_vl_model(
        Agent_stage1_system_prompt,
        f"Image titles: {img_titles}\n\nHere is the question: {q}",
        image_paths=img_paths if img_paths else None,
    )

# ── STAGE 2 ──────────────────────────────────────────────────────────────────
# Cross-modal agents receive real Stage 1 insight OR the UNAVAILABLE_MSG
# so they know not to treat missing data as contradictory evidence

def text_cross_agent(q, text_insight, tables, image_items):
    img_paths   = get_image_paths(image_items)
    table_block = cap(tables, 1000) if tables.strip() else UNAVAILABLE_MSG
    return run_vl_model(
        Agent_stage2_system_prompt,
        f"Text insight:\n{cap(text_insight, 1500)}\n\n"
        f"Question: {q}\n\nTable:\n{table_block}",
        image_paths=img_paths if img_paths else None,
    )

def table_cross_agent(q, text, table_insight, image_items):
    img_paths  = get_image_paths(image_items)
    text_block = cap(text, 1000) if text.strip() and text != "No text" else UNAVAILABLE_MSG
    return run_vl_model(
        Agent_stage2_system_prompt,
        f"Table insight:\n{cap(table_insight, 1500)}\n\n"
        f"Question: {q}\n\nText:\n{text_block}",
        image_paths=img_paths if img_paths else None,
    )

def image_cross_agent(q, text, tables, image_insight, image_items):
    img_paths   = get_image_paths(image_items)
    text_block  = cap(text,   800) if text.strip()   and text   != "No text" else UNAVAILABLE_MSG
    table_block = cap(tables, 800) if tables.strip()                          else UNAVAILABLE_MSG
    return run_vl_model(
        Agent_stage2_system_prompt,
        f"Image insight:\n{cap(image_insight, 1500)}\n\n"
        f"Question: {q}\n\nText:\n{text_block}\n\nTable:\n{table_block}",
        image_paths=img_paths if img_paths else None,
    )

# ── STAGE 3 ──────────────────────────────────────────────────────────────────

def aggregator_agent(q, text_cross, table_cross, image_cross, with_question=True):
    q_line = f"Question: {q}\n\n" if with_question else ""
    return run_vl_model(
        Agent_stage3_system_prompt,
        f"{q_line}"
        f"Text-anchored synthesis:\n{cap(text_cross,  1000)}\n\n"
        f"Table-anchored synthesis:\n{cap(table_cross, 1000)}\n\n"
        f"Image-anchored synthesis:\n{cap(image_cross, 1000)}"
    )

print("✅ All 7 agent functions ready")
print("   Unavailable modalities marked with UNAVAILABLE_MSG — not treated as contradictions")

In [ ]:
# ============================================================
# Cell 11 — Full MAMMQA pipeline
# Requirement 1: routing based on DATA AVAILABILITY only, not question type
# ============================================================

def run_mammqa(ex, verbose=True, with_question_in_aggregator=True):
    q      = ex['question']
    text   = ex['text']
    table  = ex['table']
    images = ex['image_items']
    q_type = ex.get('type', '')       # kept for logging/reporting ONLY

    if verbose:
        print(f"\n{'='*65}")
        print(f"Q: {q}")
        print(f"Type: {q_type} (for reporting only — NOT used to skip agents)")
        print(f"{'='*65}")

    result = {"question": q, "type": q_type, "stage1": {}, "stage2": {}}

    # ── AVAILABILITY CHECK — based purely on data, not question type ─────────
    # This is Requirement 1: the pipeline never looks at q_type to decide
    # which agents to run
    run_text  = bool(text.strip())          and text  != "No text"
    run_table = bool(table.strip())
    run_image = len(get_image_paths(images)) > 0

    if verbose:
        print(f"  Data available: text={run_text} | table={run_table} | image={run_image}")

    # ── STAGE 1 ─────────────────────────────────────────────────────────────
    if verbose: print("\n▶ Stage 1: Modality Expert Agents...")

    # Run agents for available modalities; record structured status
    ti_output = text_agent(q, text)   if run_text  else None
    bi_output = table_agent(q, table)  if run_table else None
    ii_output = image_agent(q, images) if run_image else None

    # Structured stage1 with availability status (Requirement 1)
    result['stage1'] = {
        "text":  {"status": "available",   "output": ti_output} if run_text  else {"status": "unavailable", "output": None},
        "table": {"status": "available",   "output": bi_output} if run_table else {"status": "unavailable", "output": None},
        "image": {"status": "available",   "output": ii_output} if run_image else {"status": "unavailable", "output": None},
    }

    # For Stage 2 cross-agents: use real output OR UNAVAILABLE_MSG — never "skipped"
    ti = ti_output if run_text  else UNAVAILABLE_MSG
    bi = bi_output if run_table else UNAVAILABLE_MSG
    ii = ii_output if run_image else UNAVAILABLE_MSG

    if verbose:
        print(f"  Text:  {'✅ ' + extract_and_clean(ti)[:70] if run_text  else '⬜ unavailable'}")
        print(f"  Table: {'✅ ' + extract_and_clean(bi)[:70] if run_table else '⬜ unavailable'}")
        print(f"  Image: {'✅ ' + extract_and_clean(ii)[:70] if run_image else '⬜ unavailable'}")

    # ── STAGE 2 ─────────────────────────────────────────────────────────────
    if verbose: print("\n▶ Stage 2: Cross-Modal Synthesis...")

    tc = text_cross_agent(q, ti, table, images)
    bc = table_cross_agent(q, text, bi, images)
    ic = image_cross_agent(q, text, table, ii, images)

    result['stage2'] = {"text": tc, "table": bc, "image": ic}

    if verbose:
        print(f"  ✅ Text-anchored:  {extract_and_clean(tc)[:70]}")
        print(f"  ✅ Table-anchored: {extract_and_clean(bc)[:70]}")
        print(f"  ✅ Image-anchored: {extract_and_clean(ic)[:70]}")

    # ── STAGE 3 ─────────────────────────────────────────────────────────────
    if verbose: print("\n▶ Stage 3: Aggregator...")

    final_raw    = aggregator_agent(q, tc, bc, ic, with_question=with_question_in_aggregator)
    final_answer = extract_and_clean(final_raw)   # Requirement 2: short answer

    result['final_raw']    = final_raw
    result['final_answer'] = final_answer
    result['used_mods']    = {'text': run_text, 'table': run_table, 'image': run_image}
    result['with_q']       = with_question_in_aggregator

    if verbose:
        print(f"\n  RAW OUTPUT:   {final_raw[:120]}")
        print(f"  FINAL ANSWER: {final_answer}")
    return result

print("✅ MAMMQA pipeline ready")
print("   Routing: data availability only (no question-type peeking)")
print("   Answers: extract_and_clean() enforces short span output")

In [ ]:
# ============================================================
# Cell 12 — Evaluation + Baseline functions
# ============================================================

def normalize(text):
    return re.sub(r'\W+', ' ', str(text).lower().strip()).split()

def exact_match(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    return normalize(pred) == normalize(g)

def token_f1(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    p, g = normalize(pred), normalize(g)
    if not p or not g: return 0.0
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    pr, rc = n/len(p), n/len(g)
    return 2*pr*rc/(pr+rc)

def evaluate(rows):
    em = [exact_match(r['prediction'], r['gold_answer']) for r in rows]
    f1 = [token_f1(r['prediction'],   r['gold_answer']) for r in rows]
    return {'EM%': round(sum(em)/len(em)*100,1) if em else 0,
            'F1%': round(sum(f1)/len(f1)*100,1) if f1 else 0, 'n': len(rows)}

def save_jsonl(rows, path):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False)+'\n')

# ── Baselines ─────────────────────────────────────────────────────────────────

def zero_shot(q):
    raw = run_vl_model("Answer as concisely as possible.", f"Question: {q}\nAnswer:")
    return extract_answer(raw)

def cot_baseline(q, text, table, image_items):
    img_titles = ", ".join(i['title'] for i in image_items if i.get('title')) or "none"
    img_paths  = get_image_paths(image_items)
    raw = run_vl_model(
        "Use the provided text, table, and images. Think step by step. "
        "End with: <answer> your answer </answer>",
        f"Question: {q}\n\nText:\n{cap(text,1500)}\n\nTable:\n{cap(table,1000)}\n\nImages: {img_titles}",
        image_paths=img_paths if img_paths else None,
    )
    return extract_answer(raw)

print("✅ Evaluation and baseline functions ready")

In [ ]:
# ============================================================
# Cell 13 — Gap 6: Lightweight Tree-of-Thoughts
# ============================================================

def run_tot(q, text, table, image_items, k=3):
    """
    Gap 6 fix: Lightweight ToT.
    Generates k candidate answers independently, then synthesizes.
    """
    img_paths = get_image_paths(image_items)
    img_titles = ", ".join(i['title'] for i in image_items if i.get('title')) or "none"

    # Generate k independent thought candidates
    thoughts = []
    for ki in range(k):
        thought = run_vl_model(
            "You are a careful reasoning agent. Given the context, think through "
            "the question and provide your best answer with confidence (0.0-1.0). "
            "Format: <thought> reasoning </thought> <confidence> score </confidence> "
            "<answer> answer </answer>",
            f"Question: {q}\n\nText:\n{cap(text,1200)}\n\nTable:\n{cap(table,800)}\n\nImages: {img_titles}",
            image_paths=img_paths if img_paths else None,
        )
        thoughts.append(thought)

    # Extract confidence scores and answers
    scored = []
    for t in thoughts:
        conf_m = re.search(r'<confidence>(.*?)</confidence>', t, re.IGNORECASE)
        try:    conf = float(conf_m.group(1).strip()) if conf_m else 0.5
        except: conf = 0.5
        ans = extract_answer(t)
        scored.append((conf, ans, t))

    # Select highest confidence answer
    best_conf, best_ans, best_thought = max(scored, key=lambda x: x[0])

    return {
        "final_answer": best_ans,
        "best_confidence": best_conf,
        "thoughts": thoughts,
    }

print("✅ Tree-of-Thoughts function ready (k=3 candidates per question)")

In [ ]:
# ============================================================
# Cell 14 — Gap 5: Robustness testing helpers
# ============================================================

IRRELEVANT_CONTEXT = """The Eiffel Tower is located in Paris, France. 
It was built by Gustave Eiffel between 1887 and 1889. 
The tower stands 330 metres tall and was the world's tallest structure until 1930. 
It receives approximately 7 million visitors per year."""

def shuffle_text(text):
    """Gap 5: Scramble sentence order to break contextual grounding."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= 1: return text
    random.shuffle(sentences)
    return ' '.join(sentences)

def inject_irrelevant(text):
    """Gap 5: Append unrelated text to test noise robustness."""
    return text + "\n\n[Additional context]: " + IRRELEVANT_CONTEXT

def run_robustness(ex):
    """Run MAMMQA under two adversarial text perturbations."""
    results = {}

    # Test 1: Text shuffle
    shuffled_ex = dict(ex)
    shuffled_ex['text'] = shuffle_text(ex['text'])
    res_shuffle = run_mammqa(shuffled_ex, verbose=False)
    results['text_shuffle'] = {
        'prediction':  res_shuffle['final_answer'],
        'exact_match': exact_match(res_shuffle['final_answer'], ex['gold_answer']),
        'token_f1':    token_f1(res_shuffle['final_answer'], ex['gold_answer']),
    }

    # Test 2: Irrelevant context injection
    injected_ex = dict(ex)
    injected_ex['text'] = inject_irrelevant(ex['text'])
    res_inject = run_mammqa(injected_ex, verbose=False)
    results['irrelevant_context'] = {
        'prediction':  res_inject['final_answer'],
        'exact_match': exact_match(res_inject['final_answer'], ex['gold_answer']),
        'token_f1':    token_f1(res_inject['final_answer'], ex['gold_answer']),
    }

    return results

print("✅ Robustness testing functions ready")
print("   Text shuffle: scrambles sentence order")
print("   Irrelevant context: injects unrelated paragraph")

In [ ]:
# ============================================================
# Cell 15 — Run MAMMQA pipeline (main + question-agnostic ablation)
# ============================================================

mammqa_rows    = []
agnostic_rows  = []

for run_i, ex in enumerate(examples, 1):
    print(f"\n{'#'*65}")
    print(f"  MAMMQA {run_i}/{len(examples)} | Type: {ex['type']}")
    print(f"{'#'*65}")

    # Standard MAMMQA (aggregator sees the question)
    result = run_mammqa(ex, verbose=True, with_question_in_aggregator=True)

    row = {
        "id": ex["id"], "type": ex["type"],
        "question": ex["question"], "gold_answer": ex["gold_answer"],
        "prediction": result["final_answer"],
        "stage1": result["stage1"], "stage2": result["stage2"],
        "final_raw": result["final_raw"],
    }
    row["exact_match"] = exact_match(row["prediction"], row["gold_answer"])
    row["token_f1"]    = token_f1(row["prediction"],   row["gold_answer"])
    mammqa_rows.append(row)

    # Gap 7: Question-agnostic aggregator ablation
    if RUN_AGNOSTIC_ABLATION:
        print("\n  ▶ Running question-agnostic ablation...")
        agn_result = run_mammqa(ex, verbose=False, with_question_in_aggregator=False)
        agn_row = {
            "id": ex["id"], "type": ex["type"],
            "question": ex["question"], "gold_answer": ex["gold_answer"],
            "prediction": agn_result["final_answer"],
        }
        agn_row["exact_match"] = exact_match(agn_row["prediction"], agn_row["gold_answer"])
        agn_row["token_f1"]    = token_f1(agn_row["prediction"],   agn_row["gold_answer"])
        agnostic_rows.append(agn_row)
        print(f"     Agnostic answer: {agn_row['prediction']}")

    print(f"\n  GOLD:  {row['gold_answer']}")
    print(f"  PRED:  {row['prediction']}")
    print(f"  EM: {'✅' if row['exact_match'] else '❌'}  F1: {row['token_f1']:.3f}")

# Save
save_jsonl(mammqa_rows, OUTPUT_DIR / "mammqa_predictions.jsonl")

mammqa_eval  = evaluate([{"prediction": r["prediction"], "gold_answer": r["gold_answer"]} for r in mammqa_rows])
agnostic_eval = evaluate([{"prediction": r["prediction"], "gold_answer": r["gold_answer"]} for r in agnostic_rows]) if agnostic_rows else {}

print(f"\n{'='*65}")
print(f"  MAMMQA Results: EM={mammqa_eval['EM%']}%  F1={mammqa_eval['F1%']}%")
if agnostic_eval:
    print(f"  Question-agnostic: EM={agnostic_eval['EM%']}%  F1={agnostic_eval['F1%']}%")
    delta = agnostic_eval['EM%'] - mammqa_eval['EM%']
    print(f"  Agnostic vs Standard: {'+' if delta>=0 else ''}{delta:.1f}% (paper found +6.76%)")
print(f"{'='*65}")

In [ ]:
# ============================================================
# Cell 16 — Run baselines (Zero-shot and CoT)
# ============================================================

baseline_rows = []

if RUN_BASELINES:
    for run_i, ex in enumerate(examples, 1):
        print(f"\n{'─'*65}")
        print(f"BASELINES {run_i}/{len(examples)} | {ex['type']} | {ex['question']}")

        zs  = zero_shot(ex["question"])
        cot = cot_baseline(ex["question"], ex["text"], ex["table"], ex["image_items"])

        row = {
            "id": ex["id"], "type": ex["type"],
            "question": ex["question"], "gold_answer": ex["gold_answer"],
            "zs_prediction": zs, "cot_prediction": cot,
            "zs_em":  exact_match(zs,  ex["gold_answer"]),
            "zs_f1":  token_f1(zs,     ex["gold_answer"]),
            "cot_em": exact_match(cot, ex["gold_answer"]),
            "cot_f1": token_f1(cot,    ex["gold_answer"]),
        }
        baseline_rows.append(row)
        print(f"  Gold: {row['gold_answer']}")
        print(f"  ZS:   {row['zs_prediction']}  | EM:{'✅' if row['zs_em'] else '❌'} F1:{row['zs_f1']:.3f}")
        print(f"  CoT:  {row['cot_prediction']}  | EM:{'✅' if row['cot_em'] else '❌'} F1:{row['cot_f1']:.3f}")

    save_jsonl(baseline_rows, OUTPUT_DIR / "baselines.jsonl")
    pd.DataFrame(baseline_rows).to_csv(OUTPUT_DIR / "baselines.csv", index=False)

    zs_eval  = evaluate([{"prediction": r["zs_prediction"],  "gold_answer": r["gold_answer"]} for r in baseline_rows])
    cot_eval = evaluate([{"prediction": r["cot_prediction"], "gold_answer": r["gold_answer"]} for r in baseline_rows])

    print(f"\nZero-shot: EM={zs_eval['EM%']}%  F1={zs_eval['F1%']}%")
    print(f"CoT:       EM={cot_eval['EM%']}%  F1={cot_eval['F1%']}%")
else:
    print("RUN_BASELINES=False")

In [ ]:
# ============================================================
# Cell 17 — Gap 6: Tree-of-Thoughts
# ============================================================

tot_rows = []

if RUN_TOT:
    print("Running Tree-of-Thoughts (k=3 candidates per question)...")
    for run_i, ex in enumerate(examples, 1):
        print(f"\n{'─'*65}")
        print(f"ToT {run_i}/{len(examples)} | {ex['type']} | {ex['question']}")

        tot_result = run_tot(
            ex["question"], ex["text"], ex["table"], ex["image_items"], k=3
        )

        row = {
            "id": ex["id"], "type": ex["type"],
            "question": ex["question"], "gold_answer": ex["gold_answer"],
            "prediction": tot_result["final_answer"],
            "best_confidence": tot_result["best_confidence"],
        }
        row["exact_match"] = exact_match(row["prediction"], row["gold_answer"])
        row["token_f1"]    = token_f1(row["prediction"],   row["gold_answer"])
        tot_rows.append(row)

        print(f"  Gold: {row['gold_answer']}")
        print(f"  ToT:  {row['prediction']} (conf={row['best_confidence']:.2f})")
        print(f"  EM: {'✅' if row['exact_match'] else '❌'}  F1: {row['token_f1']:.3f}")

    save_jsonl(tot_rows, OUTPUT_DIR / "tot_predictions.jsonl")
    tot_eval = evaluate([{"prediction": r["prediction"], "gold_answer": r["gold_answer"]} for r in tot_rows])
    print(f"\nToT: EM={tot_eval['EM%']}%  F1={tot_eval['F1%']}%")
else:
    print("RUN_TOT=False")

In [ ]:
# ============================================================
# Cell 18 — Gap 5: Robustness testing
# ============================================================

robustness_rows = []

if RUN_ROBUSTNESS:
    print("Running robustness tests (text shuffle + irrelevant context)...")
    for run_i, ex in enumerate(examples, 1):
        print(f"\n{'─'*65}")
        print(f"Robustness {run_i}/{len(examples)} | {ex['type']}")
        print(f"Q: {ex['question']}")

        normal_pred = mammqa_rows[run_i-1]['prediction'] if run_i <= len(mammqa_rows) else "N/A"
        rob_results = run_robustness(ex)

        row = {
            "id": ex["id"], "type": ex["type"],
            "question": ex["question"], "gold_answer": ex["gold_answer"],
            "normal_prediction":  normal_pred,
            "shuffle_prediction": rob_results['text_shuffle']['prediction'],
            "inject_prediction":  rob_results['irrelevant_context']['prediction'],
            "normal_f1":  token_f1(normal_pred, ex["gold_answer"]),
            "shuffle_f1": rob_results['text_shuffle']['token_f1'],
            "inject_f1":  rob_results['irrelevant_context']['token_f1'],
        }
        robustness_rows.append(row)

        print(f"  Normal:   {row['normal_prediction']} (F1={row['normal_f1']:.3f})")
        print(f"  Shuffled: {row['shuffle_prediction']} (F1={row['shuffle_f1']:.3f})")
        print(f"  Injected: {row['inject_prediction']} (F1={row['inject_f1']:.3f})")
        print(f"  Shuffle drop: {(row['normal_f1']-row['shuffle_f1'])*100:.1f}% F1")
        print(f"  Inject drop:  {(row['normal_f1']-row['inject_f1'])*100:.1f}% F1")

    save_jsonl(robustness_rows, OUTPUT_DIR / "robustness.jsonl")
    pd.DataFrame(robustness_rows).to_csv(OUTPUT_DIR / "robustness.csv", index=False)
    print("\n✅ Robustness results saved")
else:
    print("RUN_ROBUSTNESS=False")

In [ ]:
# ============================================================
# Cell 19 — Final comparison table (all methods)
# ============================================================

print(f"\n{'='*65}")
print("  FINAL RESULTS — ALL METHODS")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE} | Top-p: {TOP_P}")
print(f"  GPUs: {torch.cuda.device_count()} x T4")
print(f"{'='*65}")
print(f"{'Method':<28} {'EM%':>6} {'F1%':>6}")
print(f"{'─'*40}")

if baseline_rows:
    print(f"{'Zero-shot':<28} {zs_eval['EM%']:>6} {zs_eval['F1%']:>6}")
    print(f"{'CoT':<28} {cot_eval['EM%']:>6} {cot_eval['F1%']:>6}")

if tot_rows:
    print(f"{'Tree-of-Thoughts':<28} {tot_eval['EM%']:>6} {tot_eval['F1%']:>6}")

print(f"{'MAMMQA (with Q in agg)':<28} {mammqa_eval['EM%']:>6} {mammqa_eval['F1%']:>6}")

if agnostic_eval:
    print(f"{'MAMMQA (Q-agnostic agg)':<28} {agnostic_eval['EM%']:>6} {agnostic_eval['F1%']:>6}")

print(f"{'='*40}")

if robustness_rows:
    avg_shuffle_drop = sum(r['normal_f1']-r['shuffle_f1'] for r in robustness_rows)/len(robustness_rows)*100
    avg_inject_drop  = sum(r['normal_f1']-r['inject_f1']  for r in robustness_rows)/len(robustness_rows)*100
    print(f"\nRobustness (avg F1 drop):")
    print(f"  Text shuffle:     -{avg_shuffle_drop:.1f}% (paper: -91.24% at 7B)")
    print(f"  Irrelevant inject: -{avg_inject_drop:.1f}% (paper: -5.65% at 7B)")

print(f"\nOutput files saved to: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("*.jsonl")) + sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")